# PAATRA Step 3b — Part 2/5: Train Config A (A_80K_inherited)

Trains the **A_80K_inherited** student. Loads the chunks and vocabulary artifacts saved by `03_scaled_setup.ipynb`, runs distillation, then saves the trained student bundle to Drive.

**Prerequisite:** run `03_scaled_setup.ipynb` first.

The completed checkpoint for this configuration was not recovered. This notebook is included so the baseline can be reproduced.


In [ ]:
!pip install -q torch transformers accelerate datasets

In [ ]:
import torch, math, time, os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, GPT2Config, GPT2LMHeadModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEACHER_ID = "Qwen/Qwen2.5-0.5B"
PRESET = "FULL"

if PRESET == "FULL":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 512, 4, 15000, 80_000
elif PRESET == "FAST":
    SEQ_LEN, BATCH_SIZE, NUM_TRAIN_STEPS, NUM_CORPUS_SAMPLES = 256, 8, 8000, 50_000

LEARNING_RATE = 3e-4
WARMUP_STEPS = 500
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.7


## 1. Mount Drive and load shared artifacts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/paatra'
os.makedirs(f'{DRIVE_DIR}/students', exist_ok=True)
NAME = "A_80K_inherited"

meta = torch.load(f'{DRIVE_DIR}/meta.pt', weights_only=False)
chunks = torch.load(f'{DRIVE_DIR}/chunks.pt', weights_only=False)
vocabs = torch.load(f'{DRIVE_DIR}/vocabs.pt', weights_only=False)
assert meta['SEQ_LEN'] == SEQ_LEN
assert meta['NUM_TRAIN_STEPS'] == NUM_TRAIN_STEPS
s2t = vocabs[NAME]['s2t']
t2s = vocabs[NAME]['t2s']
config_dict = vocabs[NAME]['config']
print(f'Vocab: {len(s2t):,} | Config: {config_dict}')


## 2. Load teacher

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_ID, torch_dtype=torch.float16, device_map=DEVICE,
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False


## 3. Define student and training loop

In [ ]:
def create_student(config_dict, vocab_size):
    config = GPT2Config(
        vocab_size=vocab_size, n_embd=config_dict['n_embd'],
        n_layer=config_dict['n_layer'], n_head=config_dict['n_head'],
        n_inner=4 * config_dict['n_embd'], activation_function='gelu_new',
        resid_pdrop=0.1, embd_pdrop=0.1, attn_pdrop=0.1,
        n_positions=SEQ_LEN, bos_token_id=None, eos_token_id=None, pad_token_id=None,
    )
    return GPT2LMHeadModel(config)

def cosine_lr(step, warmup, total, base_lr):
    if step < warmup:
        return base_lr * step / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

class KDDataset(Dataset):
    def __init__(self, chunks, t2s):
        self.chunks = chunks; self.t2s = t2s
    def __len__(self): return len(self.chunks)
    def __getitem__(self, idx):
        teacher_ids = self.chunks[idx].tolist()
        student_ids = [self.t2s.get(tid, 0) for tid in teacher_ids]
        loss_mask = [1.0 if tid in self.t2s else 0.0 for tid in teacher_ids]
        return {
            'teacher_ids': torch.tensor(teacher_ids, dtype=torch.long),
            'student_ids': torch.tensor(student_ids, dtype=torch.long),
            'loss_mask': torch.tensor(loss_mask, dtype=torch.float),
        }

def train_student():
    student_vocab_size = len(s2t)
    s2t_tensor = torch.tensor(s2t, dtype=torch.long, device=DEVICE)
    student = create_student(config_dict, student_vocab_size).to(DEVICE)
    print(f'Total parameters: {sum(p.numel() for p in student.parameters()):,}')
    dataloader = DataLoader(KDDataset(chunks, t2s), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    step = 0; loss_history = []; running_loss = running_kd = running_ce = 0.0; start = time.time()
    student.train()
    while step < NUM_TRAIN_STEPS:
        for batch in dataloader:
            if step >= NUM_TRAIN_STEPS: break
            lr = cosine_lr(step, WARMUP_STEPS, NUM_TRAIN_STEPS, LEARNING_RATE)
            for pg in optimizer.param_groups: pg['lr'] = lr
            teacher_ids = batch['teacher_ids'].to(DEVICE)
            student_ids = batch['student_ids'].to(DEVICE)
            loss_mask = batch['loss_mask'].to(DEVICE)
            assert student_ids.max().item() < student_vocab_size
            with torch.no_grad():
                t_logits = teacher(input_ids=teacher_ids).logits.float()
                t_subset = t_logits[:, :-1, :].index_select(2, s2t_tensor)
            s_logits = student(input_ids=student_ids).logits[:, :-1, :]
            labels = student_ids[:, 1:]; mask = loss_mask[:, 1:]; mask_sum = mask.sum().clamp(min=1)
            t_probs = F.softmax(t_subset / KD_TEMPERATURE, dim=-1)
            s_log_probs = F.log_softmax(s_logits / KD_TEMPERATURE, dim=-1)
            kd_loss = (F.kl_div(s_log_probs, t_probs, reduction='none').sum(-1) * mask).sum() / mask_sum * (KD_TEMPERATURE ** 2)
            ce = F.cross_entropy(s_logits.reshape(-1, s_logits.size(-1)), labels.reshape(-1), reduction='none').reshape(labels.shape)
            ce_loss = (ce * mask).sum() / mask_sum
            loss = KD_ALPHA * kd_loss + (1 - KD_ALPHA) * ce_loss
            optimizer.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0); optimizer.step()
            running_loss += loss.item(); running_kd += kd_loss.item(); running_ce += ce_loss.item(); step += 1
            if step % 1000 == 0:
                entry = {'step': step, 'loss': running_loss/1000, 'kd': running_kd/1000, 'ce': running_ce/1000}
                loss_history.append(entry); print(entry); running_loss = running_kd = running_ce = 0.0
    student.eval(); return student, loss_history


## 4. Train and save

In [ ]:
out_path = f'{DRIVE_DIR}/students/{NAME}.pt'
if os.path.exists(out_path):
    print(f'Checkpoint already exists: {out_path}')
else:
    student, losses = train_student()
    bundle = {
        'name': NAME, 'state_dict': student.state_dict(),
        'model_config': student.config.to_dict(), 'config': config_dict,
        's2t': s2t, 't2s': t2s, 'losses': losses,
        'preset': PRESET, 'seq_len': SEQ_LEN, 'num_train_steps': NUM_TRAIN_STEPS,
    }
    torch.save(bundle, out_path)
    print(f'Saved: {out_path}')
